In [7]:
!pip install psycopg2-binary pandas -q
print("Installed!")

Installed!



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import psycopg2

In [8]:
conn=psycopg2.connect(
    host="localhost",
    dbname="education",
    user="postgres",
    password="shivani@",
    port=5432
)
cur=conn.cursor()
print("connected sucessfully")

connected sucessfully


In [10]:
cur.execute("""
    CREATE TABLE courses (
        id SERIAL PRIMARY KEY,
        course_name VARCHAR(50)
    );

    CREATE TABLE students (
        id SERIAL PRIMARY KEY,
        name VARCHAR(50),
        marks INT
    );

    CREATE TABLE enrollments (
        student_id INT REFERENCES students(id),
        course_id INT REFERENCES courses(id)
    );
""")


In [11]:
conn.commit()
print("Tables created")

Tables created


In [12]:
cur.execute("INSERT INTO courses (course_name) values (%s)",("Math",))

cur.executemany("INSERT INTO courses (course_name) values (%s)",[("Science",),("Music",),("Art",)])

print("inserted sucessfully")

inserted sucessfully


In [14]:
cur.execute("INSERT INTO courses (course_name) values (%s)",("Math",))
cur.execute("INSERT INTO courses (course_name) values (%s)",("English",))

In [15]:
conn.commit()

In [25]:
conn.rollback()

In [26]:
cur.executemany("INSERT INTO students (name,marks) values (%s,%s)",[("Shivani",95),("Ajay",85),("Pavani",47),("Divyanshi",59),("Jyothi",69)])

In [27]:
conn.commit()

In [29]:
cur.executemany("INSERT INTO enrollments values (%s,%s)",[(1,2),(1,4),(2,5),(3,2),(5,1)])
conn.commit()

In [31]:
cur.execute("Delete from enrollments")
conn.commit()

In [32]:
cur.executemany("INSERT INTO enrollments values (%s,%s)",[(1,2),(1,4),(2,5),(3,2),(5,1)])
conn.commit()

In [34]:
cur.execute("Select * from students")
rows=cur.fetchall()

print(rows)
print("-"*30)

for row in rows:
    print(f"id={row[0]}  name={row[1]}  marks={row[2]}")

[(1, 'Shivani', 95), (2, 'Ajay', 85), (3, 'Pavani', 47), (4, 'Divyanshi', 59), (5, 'Jyothi', 69)]
------------------------------
id=1  name=Shivani  marks=95
id=2  name=Ajay  marks=85
id=3  name=Pavani  marks=47
id=4  name=Divyanshi  marks=59
id=5  name=Jyothi  marks=69


In [38]:
cur.execute("Select * from students")
rows=cur.fetchone()
print(rows)

(1, 'Shivani', 95)


In [45]:
conn.rollback()

In [51]:
cur.execute("select * from Students where marks>75 order by name")
higher=cur.fetchall()
print(higher)

[(2, 'Ajay', 85), (1, 'Shivani', 95)]


In [52]:
import pandas as pd
df=pd.read_sql("Select * from students",conn)
df

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16480\1065339187.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql("Select * from students",conn)


,id,name,marks
0,1,Shivani,95
1,2,Ajay,85
2,3,Pavani,47
3,4,Divyanshi,59
4,5,Jyothi,69


In [ ]:
def add_student(name, marks):
    cur.execute(
        "INSERT INTO students (name, marks) VALUES (%s, %s) RETURNING id",
        (name, marks)
    )
    new_id = cur.fetchone()[0]     # RETURNING gives us the new id back!
    conn.commit()
    print(f"Added {name} with id {new_id} ✅")

add_student("Zara", 88)
add_student("Arjun", 74)


Added Zara with id 6 ✅
Added Arjun with id 7 ✅


In [63]:
conn.rollback()

In [66]:
conn.rollback()

In [67]:
def  add_courses(name):
    cur.execute("INSERT INTO courses (course_name) values (%s) RETURNING id",(name,))
    new_id=cur.fetchone()[0]
    conn.commit()
    print(f"added {name} with id {new_id}")

add_courses("Science",) 

added Science with id 7


In [84]:
conn.rollback()

In [90]:
def enroll(name, course_name):
    cur.execute("""
        INSERT INTO enrollments (student_id, course_id)
        VALUES (
            (SELECT id FROM students WHERE name = %s),
            (SELECT id FROM courses WHERE course_name = %s)
        )
    """, (name, course_name))

    conn.commit()
    print(f"{name} enrolled in {course_name}")


conn.rollback()

enroll("Dev","Physics")

Dev enrolled in Physics


In [95]:
import pandas as pd

def report_card(student_name):
    cur.execute("""
        SELECT c.course_name, s.marks
        FROM students s
        JOIN enrollments e
            ON s.id = e.student_id
        JOIN courses c
            ON e.course_id = c.id
        WHERE s.name = %s
    """, (student_name,))

    rows = cur.fetchall()

    df = pd.DataFrame(rows, columns=["course_name", "marks"])

    print(f"\nReport Card for {student_name}")
    print(df)

report_card("Shivani")


Report Card for Shivani
  course_name  marks
0     Science     95
1         Art     95


In [97]:
name = input("Enter student name: ")
marks = int(input("Enter marks: "))

cur.execute(
    "INSERT INTO students (name, marks) VALUES (%s, %s)",
    (name, marks)
)

conn.commit()

print(f"{name} added successfully with marks {marks}")

Ravi added successfully with marks 87


In [ ]:
cur.close()
conn.close()
print("Disconnected. Goodbye PostgreSQL! 👋")